# Parda — Phase 1: Generate synthetic Indian documents (CPU only)

**Before running:** Settings → Accelerator: **None** (CPU, no GPU quota used) · Internet: **On**
Secrets needed (Add-ons → Secrets): `GITHUB_TOKEN` (repo is private), `HF_TOKEN` (only for the optional backup cell).

**How to use:** run cells 1–4 interactively to check the previews. If they look good,
click **Save Version → Save & Run All**, close the tab, and come back in ~90 min.

In [ ]:
# 1. Get the code (cloned to /tmp so the token never ends up in saved outputs)
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
REPO = "chetanjagan/Parda"
CODE = "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Fonts (Hindi + Kannada) and packages. Must show RAQM: True and all fonts True
!bash scripts/setup_kaggle.sh

In [ ]:
# 3. Tests: every line should say ok
!python tests/test_synth.py

In [ ]:
# 4. Small preview run: look at these images before the big run
!rm -rf /tmp/preview && python -m parda.synth.generate --n 40 --out /tmp/preview --workers 4 > /dev/null
!python -m parda.synth.visualize --data /tmp/preview --k 8
import glob
from IPython.display import Image as IPImage, display
for f in sorted(glob.glob("/tmp/preview/previews/*.jpg"))[:8]:
    print(os.path.basename(f)); display(IPImage(f, width=520))

In [ ]:
# 5. Full run (~75-90 min on 4 CPU cores). Saved to /kaggle/working -> becomes notebook output
N = 30000
OUT = "/kaggle/working/parda_synth_v1"
!python -m parda.synth.generate --n {N} --out {OUT} --workers 4 --seed 42
!du -sh {OUT}

In [ ]:
# 6. (Optional) Backup to a private Hugging Face dataset, packed into tar shards (faster upload)
BACKUP_TO_HF = False
if BACKUP_TO_HF:
    import tarfile, shutil
    from huggingface_hub import HfApi
    api = HfApi(token=secrets.get_secret("HF_TOKEN"))
    user = api.whoami()["name"]
    repo_id = f"{user}/parda-synth-v1"
    api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)
    imgs = sorted(glob.glob(f"{OUT}/images/*.jpg"))
    os.makedirs("/tmp/shards", exist_ok=True)
    for k in range(0, len(imgs), 5000):
        with tarfile.open(f"/tmp/shards/images_{k // 5000:03d}.tar", "w") as t:
            for f in imgs[k:k + 5000]:
                t.add(f, arcname="images/" + os.path.basename(f))
    for f in glob.glob(f"{OUT}/*.json*"):
        shutil.copy(f, "/tmp/shards/")
    api.upload_folder(folder_path="/tmp/shards", repo_id=repo_id, repo_type="dataset")
    print("uploaded to", repo_id)

## After the committed run finishes
Open this notebook's **Output** tab → **New Dataset** → name it `parda-synth-v1`.
Every later notebook (OCR, GLiNER, YOLO) just attaches that dataset as Input.